# Transaction Foundation Model on Ray — Part 5: Batch embedding extraction

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: ~15 min on the GPU workers)

---

Part 4 trained the foundation model. This notebook runs it over every transaction and stores the embeddings for the downstream models to read. In production this is the job that recurs: new transactions arrive, embeddings refresh on a schedule. The work is heterogeneous — CPU to tokenize, GPU for the forward pass — and it runs as one streaming Ray Data pipeline.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import logging

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},
         logging_level=logging.ERROR)

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts; full = every card
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

## Turn the trained model into one vector per transaction

The foundation model pretrained on long histories, but the downstream classifier needs a fixed-length vector per transaction. Following NVIDIA's blueprint, each transaction is embedded on its own: feed `<bos>` + its 12 field tokens + `<eos>` through the decoder and read the hidden state at the last real token. Embedding a transaction alone means the vector is a learned re-encoding of the same 13 raw fields — what the model adds is what it learned about transactions in general during pretraining, not context from this card's history. Part 7 revisits that choice with history windows.

Not everything gets embedded. The training set is balance-sampled — every fraud plus normals up to a cap (1M rows at `full`; NVIDIA's recipe) — so the downstream classifier sees fraud often enough to learn it, without paying to embed all 19.5M train transactions. Val and test use the 100K stratified eval subsets from Part 2, embedded whole. Each split is written as `embed_<split>.npy` + `lbl_<split>.npy` + `raw_<split>.parquet` (the 13 native fields), so Part 6 fits its classifiers without touching the tokenizer again.

## Stream CPU work into GPU forward passes

This stage needs two kinds of hardware at once: tokenizing transactions is CPU work and the forward pass is GPU work. The model is also expensive to load, so it should load once per GPU and stay resident. The pipeline gives each piece its own resources:

- A **CPU task** runs the seeded, order-sensitive preamble once per split — the balanced sample and the sort that fixes output row order — and writes `lbl_`/`raw_` immediately; labels and features never needed a GPU.
- **CPU workers** stream batches of transactions into token rows (`encode_txn_batch`, the same identity-verified tokenizer path as Part 3).
- **GPU actors** do only forward passes. Passing a class (`GPUEmbedder`) to `map_batches` is what creates them: Ray Data builds an actor pool, each actor's constructor runs once and loads the model, and batches then stream through — the standard Ray Data pattern for GPU inference.

The pools scale independently: more CPU workers keep the GPUs busy, more GPU actors raise throughput linearly, and neither holds hardware the other needs. Each row carries its output position, so the assembled matrices land in the same order as the single-GPU reference — verified: labels and features byte-equal, embeddings equal to float-kernel precision (`scripts/verify_distributed_embed.py`).

In [2]:
from src.nvembed import (prepare_embed_split, encode_txn_batch, GPUEmbedder,
                         assemble_embeddings)

eb = cfg["embed"]
use_gpu = eb["use_gpu"]                    # mini: CPU actors; full: one actor per A10G
if not os.path.exists(os.path.join(paths["embeddings"], "embed_test.npy")):
    for split, fname in {"train": None, "val": "val_eval.parquet",
                         "test": "test_eval.parquet"}.items():
        # seeded preamble on one CPU task: balanced sample + order-fixing sort;
        # writes lbl_<split>.npy + raw_<split>.parquet directly
        prep = ray.get(prepare_embed_split.remote(
            paths["nvsplit"], fname, paths["embeddings"], split,
            eb["balanced_train"], 42))

        # stream: CPU workers tokenize -> GPU actors forward-pass -> shards
        shards = os.path.join(paths["embeddings"], f"_emb_{split}")
        ray.data.read_parquet(prep["prep"]) \
            .map_batches(encode_txn_batch, batch_format="pandas") \
            .map_batches(GPUEmbedder,
                         fn_constructor_kwargs={"hf_dir": paths["hf"],
                                                "batch_size": eb["batch_size"]},
                         batch_format="numpy",
                         batch_size=4096,   # rows per actor call; GPU microbatch = eb["batch_size"]
                         num_gpus=(1 if use_gpu else 0),
                         concurrency=eb["num_workers"]) \
            .write_parquet(shards)

        meta = assemble_embeddings(shards, os.path.join(paths["embeddings"],
                                                        f"embed_{split}.npy"),
                                   prep["prep"], embed_dim=cfg["model"]["d_model"])
        print(split, {**{k: prep[k] for k in ("rows", "fraud")}, **meta})
else:
    print(f"embeddings present at {paths['embeddings']} — skipping (delete the dir to re-embed)")

train {'rows': 40000, 'fraud': 2274, 'dim': 64}
val {'rows': 100000, 'fraud': 86, 'dim': 64}
test {'rows': 100000, 'fraud': 108, 'dim': 64}


## Check the embeddings for collapse

The classic self-supervised failure is silent **representation collapse**: every input maps to nearly the same vector while the training loss looks fine. Two cheap numbers guard against it — mean pairwise cosine similarity (1.0 means collapse) and mean per-feature variance (0 means collapse).

Calibration for reading them: transformer embeddings normally show high mean cosine — the vectors cluster in direction — while still separating well in the dimensions that matter, so a high mean cosine with healthy variance is typical, not near-collapse (the full-scale embeddings sit near 0.9). This check catches the degenerate case; the real test of embedding quality is the downstream comparison in Part 6.

In [3]:
# Representation-collapse check on the test embeddings: mean pairwise cosine similarity
# (→1.0 = collapse) and mean feature variance (→0 = collapse).
emb = np.load(os.path.join(paths["embeddings"], "embed_test.npy"))
lbl = np.load(os.path.join(paths["embeddings"], "lbl_test.npy"))

rng = np.random.RandomState(0)
idx = rng.choice(len(emb), size=min(2000, len(emb)), replace=False)
s = emb[idx].astype(np.float32)
s /= (np.linalg.norm(s, axis=1, keepdims=True) + 1e-8)
cos = s @ s.T
mean_cos = float((cos.sum() - len(idx)) / (len(idx) * (len(idx) - 1)))

print(f"{len(emb):,} test embeddings · dim {emb.shape[1]}  ({int(lbl.sum()):,} fraud)")
print(f"mean pairwise cosine  {mean_cos:.3f}   (→1.0 = collapse)")
print(f"mean feature variance {float(emb.var(0).mean()):.4f}   (→0 = collapse)")
print(f"example embedding[:8] = {[round(float(x), 3) for x in emb[0, :8]]}  (label {int(lbl[0])})")

100,000 test embeddings · dim 64  (108 fraud)
mean pairwise cosine  0.729   (→1.0 = collapse)
mean feature variance 0.2654   (→0 = collapse)
example embedding[:8] = [-1.843, 0.358, 0.152, -1.225, -0.139, -0.797, -0.474, -0.513]  (label 0)


## Scaling factors

Embedding cost is linear in transactions and in model size, and it is forward-pass only — no gradients, no optimizer state — so GPU memory goes to batch size and throughput rather than training bookkeeping. At `full` the stage embeds ~1.2M transactions (the 1M balanced train sample plus the two 100K eval sets) in about fifteen minutes on the GPU pool.

The knob worth understanding is the ratio between the pools. Tokenization produces rows at a rate set by the CPU worker count; each GPU actor consumes them at a rate set by the model and batch size. If the GPUs idle, add CPU workers; if tokenized batches queue, add GPU actors — the two counts move independently in the config. This is also the stage whose cost recurs: pretraining happens occasionally, but re-embedding runs every time fresh transactions arrive, so this throughput is the number a production team budgets around.

## Takeaways

Every split now has its embeddings on shared storage — `embed_/lbl_/raw_` per split — produced by one streaming pipeline: CPU workers tokenize, GPU actors run forward passes, and the two pools scale independently. That is the series' scalability claim in its most literal form — the CPU-bound step and the GPU-bound step of the same job, scaled separately. The outputs are verified against the single-GPU reference (labels and features byte-equal, embeddings to float-kernel precision), and the notebook composes the same functions the headless job runs, so the two cannot drift.

---

## Next

**Part 6 — Downstream fraud: raw vs embedding vs fusion**: fit NVIDIA's XGBoost recipe on three feature sets — the raw fields, the embedding, and both fused — and compare them at natural fraud prevalence with Average Precision (AP).